# 02 llama-server Setup and Introspection

Deep-dive into `ServerManager` — the class that launches, monitors, and
introspects the llama-server binary.

**What you will learn:**
- Start llama-server with observability endpoints
- Query `/health`, `/props`, `/slots`, `/metrics`
- Inspect loaded model properties
- Gracefully stop and restart the server

**Requirements:** Kaggle notebook with GPU T4 x2 accelerator. A GGUF model
uploaded as a Kaggle dataset.

## 1) Install

In [ ]:
!pip -q install git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1

## 2) Create a ServerManager

`ServerManager` defaults to `http://127.0.0.1:8090`. You can override the
port by passing a different `server_url`.

In [ ]:
from llamatelemetry.server import ServerManager

manager = ServerManager(server_url="http://127.0.0.1:8090")

# Point this to your uploaded GGUF dataset
model_path = "/kaggle/input/your-model/model.gguf"

## 3) Start the server with observability endpoints

Key flags:
- `enable_metrics=True` — expose Prometheus metrics at `/metrics`
- `enable_props=True` — expose model properties at `/props`
- `enable_slots=True` — expose slot states at `/slots`
- `n_parallel` — number of concurrent inference slots

In [ ]:
manager.start_server(
    model_path=model_path,
    gpu_layers=99,
    ctx_size=4096,
    n_parallel=2,
    batch_size=512,
    ubatch_size=128,
    enable_metrics=True,
    enable_props=True,
    enable_slots=True,
    verbose=True,
)

# Block until the server responds to /health
ready = manager.wait_ready(timeout=120)
print(f"Server ready: {ready}")

## 4) Query /health

In [ ]:
import json

health = manager.get_health()
print(json.dumps(health, indent=2, default=str))

## 5) Query /props — loaded model properties

In [ ]:
props = manager.get_props()
print(json.dumps(props, indent=2, default=str) if isinstance(props, dict) else props)

## 6) Query /slots — inference slot states

In [ ]:
slots = manager.get_slots()
print(json.dumps(slots, indent=2, default=str) if isinstance(slots, list) else slots)

## 7) Query /metrics — Prometheus-format metrics

The `/metrics` endpoint returns Prometheus exposition format text.

In [ ]:
metrics_text = manager.get_metrics()
if metrics_text:
    # Print first 1000 chars
    print(metrics_text[:1000])
else:
    print("No metrics available (enable_metrics may not be supported by this build)")

## 8) Server info and models

In [ ]:
info = manager.get_server_info()
print(json.dumps(info, indent=2, default=str))

## 9) Cleanup — stop the server

In [ ]:
stopped = manager.stop_server()
print(f"Server stopped: {stopped}")